# Notebook 08: Head Pose Estimation & Gaze Feasibility Study
## 3D Orientation (Yaw, Pitch, Roll) and Gaze Tracking Investigation

### Phase 2B — Physiological Signals & Temporal Analysis
This notebook investigates 3D head pose estimation from facial landmarks and evaluates the feasibility of driver gaze tracking.

**Key Objectives:**
1. Coordinate systems and kinematics for head orientation.
2. 3D Face mesh transformation matrix decomposition.
3. Euler angle extraction: Yaw (lateral rotation), Pitch (nodding), Roll (tilt).
4. Driver attention state classification (FORWARD, LEFT, RIGHT, UP, DOWN).
5. Head pose trajectory visualization over `test_video.mp4`.
6. Difficult edge cases: occlusions, head slump, glasses, passenger presence.
7. Gaze tracking feasibility study.



In [ ]:
# 1. Imports and Setup
import sys
from pathlib import Path
import csv
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import AppConfig
from src.physiological import HeadPoseEstimator

print("Head pose modules initialized.")



### 2 & 3. 3D Pose Formulation & Euler Decomposition
MediaPipe provides a $4\times4$ facial rigid transformation matrix $T = [R \mid t]$.
We extract the rotation matrix $R \in SO(3)$ and decompose it into Euler angles:
$$\text{Pitch} = \arcsin(-R_{2,0})$$
$$\text{Yaw} = \arctan2(R_{2,1}, R_{2,2})$$
$$\text{Roll} = \arctan2(R_{1,0}, R_{0,0})$$



In [ ]:
# Test HeadPoseEstimator on Canonical Transformations
estimator = HeadPoseEstimator(yaw_threshold_deg=20.0, pitch_threshold_deg=15.0, roll_threshold_deg=20.0)

# Clean identity transformation (Looking straight ahead)
identity_matrix = np.eye(4, dtype=np.float32)
res = estimator.estimate(landmarks=None, transformation_matrix=identity_matrix)

print(f"Identity Pose Result -> State: {res.state} | Yaw: {res.yaw:.1f}° | Pitch: {res.pitch:.1f}° | Roll: {res.roll:.1f}°")



### 4 & 5. Driver Attention State Classification
Angular deadbands define the forward attentional corridor:
- **`HEAD_FORWARD`**: $|\text{Yaw}| \le 20^\circ$ and $|\text{Pitch}| \le 15^\circ$
- **`HEAD_LEFT`**: $\text{Yaw} > +20^\circ$ (looking towards passenger/left mirror)
- **`HEAD_RIGHT`**: $\text{Yaw} < -20^\circ$ (looking towards driver-side mirror)
- **`HEAD_DOWN`**: $\text{Pitch} < -15^\circ$ (nodding / micro-sleep head slump)
- **`HEAD_UP`**: $\text{Pitch} > +15^\circ$ (distraction upward)



In [ ]:
# Ingest Head Pose Telemetry from Stream
telemetry_path = PROJECT_ROOT / "results" / "phase2b" / "telemetry.csv"
timestamps = []
yaws = []
pitches = []
rolls = []
states = []

with open(telemetry_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        timestamps.append(float(r["timestamp"]))
        yaws.append(float(r["yaw"]) if r["yaw"] else np.nan)
        pitches.append(float(r["pitch"]) if r["pitch"] else np.nan)
        rolls.append(float(r["roll"]) if r["roll"] else np.nan)
        states.append(r["head_state"])

# Plot 3D Orientation Trajectory
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(timestamps, yaws, label="Yaw (Turn L/R)", color="#17becf")
ax1.plot(timestamps, pitches, label="Pitch (Nod Up/Dn)", color="#bcbd22")
ax1.plot(timestamps, rolls, label="Roll (Lateral Tilt)", color="#e377c2")
ax1.set_xlabel("Time (seconds)")
ax1.set_ylabel("Degrees (°)")
ax1.set_title("Head Orientation Euler Angles")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="upper right")

from collections import Counter
state_counts = Counter(states)
sorted_counts = sorted(state_counts.items(), key=lambda x: x[1], reverse=True)
bars = ax2.bar([s[0] for s in sorted_counts], [s[1] for s in sorted_counts], color=["#1f77b4", "#ff7f0e", "#2ca02c"])
for b in bars:
    ax2.text(b.get_x() + b.get_width()/2.0, b.get_height() + 5, str(int(b.get_height())), ha="center", va="bottom")
ax2.set_ylabel("Frame Count")
ax2.set_title("Driver Attention State Distribution")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()



### 6. Edge Cases & Difficult Scenarios
1. **Head Slump / Nodding**: When fatigue causes the driver's head to droop downward, $\text{Pitch} < -15^\circ$. Correlating head droop with closed eyes ($EAR < 0.21$) creates a highly confident micro-sleep indicator.
2. **Partial Occlusion**: At steep yaw angles ($> 45^\circ$), half of the face is self-occluded.
3. **Eyeglasses**: Specular reflections can displace estimated landmark depth values.
4. **Multiple Occupants**: Passenger movements can introduce competing face bounding boxes.



### 7. Gaze Tracking Feasibility Investigation
**Objective**: Determine whether iris-based gaze tracking can be reliably added to the driver monitoring system.

#### Theoretical Approach
MediaPipe 3D Mesh includes refined iris landmarks:
- **Left Iris Center & Boundary**: Indices 468–472
- **Right Iris Center & Boundary**: Indices 473–477
By computing the normalized displacement of the pupil center relative to the eye corners (canthi):
$$\text{Gaze Ratio} = \frac{\|p_{\text{pupil}} - p_{\text{inner\_canthus}}\|}{\|p_{\text{outer\_canthus}} - p_{\text{inner\_canthus}}\|}$$

#### Hardware & Environmental Assumptions
1. Camera mounted on dashboard / steering column $\le 80$ cm from driver.
2. High input resolution (minimum $1280\times720$) to ensure iris spans $\ge 15-20$ pixels.
3. Adequate frontal illumination (or active 850nm/940nm IR illumination for night driving).

#### Limitations
1. In `test_video.mp4`, optical resolution and compression artifacts cause iris landmarks to fluctuate.
2. Tinted lenses, sunglasses, or heavy mascara prevent accurate iris boundary segmentation.
3. Gaze tracking requires per-driver calibration to determine center gaze offset.

#### Conclusion & Recommendation
Head pose estimation is robust, low-overhead, and sufficient for detecting macro-distractions and micro-sleep slumps in Phase 2B. Full gaze estimation is feasible in high-resolution hardware environments and is formally recommended for evaluation in Phase 3/4.

